# 01 — The LLM Landscape — Practical Examples

**Covers**: API connectivity tests, capability matrix, benchmark comparison, model spec explorer

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Model Spec Reference — Structured Data

In [ ]:
MODEL_SPECS = {
    # OpenAI
    'gpt-4o':               {'provider': 'OpenAI',     'context_k': 128,  'input_per_1m': 5.00,   'output_per_1m': 15.00,  'tier': 'flagship'},
    'gpt-4o-mini':          {'provider': 'OpenAI',     'context_k': 128,  'input_per_1m': 0.15,   'output_per_1m': 0.60,   'tier': 'mini'},
    'o1':                   {'provider': 'OpenAI',     'context_k': 128,  'input_per_1m': 15.00,  'output_per_1m': 60.00,  'tier': 'reasoning'},
    'o1-mini':              {'provider': 'OpenAI',     'context_k': 128,  'input_per_1m': 3.00,   'output_per_1m': 12.00,  'tier': 'reasoning'},
    # Anthropic
    'claude-3-5-sonnet':    {'provider': 'Anthropic',  'context_k': 200,  'input_per_1m': 3.00,   'output_per_1m': 15.00,  'tier': 'flagship'},
    'claude-3-5-haiku':     {'provider': 'Anthropic',  'context_k': 200,  'input_per_1m': 0.80,   'output_per_1m': 4.00,   'tier': 'mini'},
    # Google
    'gemini-1.5-pro':       {'provider': 'Google',     'context_k': 2000, 'input_per_1m': 3.50,   'output_per_1m': 10.50,  'tier': 'flagship'},
    'gemini-1.5-flash':     {'provider': 'Google',     'context_k': 1000, 'input_per_1m': 0.075,  'output_per_1m': 0.30,   'tier': 'mini'},
    'gemini-2.0-flash':     {'provider': 'Google',     'context_k': 1000, 'input_per_1m': 0.10,   'output_per_1m': 0.40,   'tier': 'mini'},
    # Open source
    'llama-3.1-70b':        {'provider': 'Meta/OSS',   'context_k': 128,  'input_per_1m': 0.52,   'output_per_1m': 0.75,   'tier': 'open-source'},
    'deepseek-v3':          {'provider': 'DeepSeek',   'context_k': 128,  'input_per_1m': 0.27,   'output_per_1m': 1.10,   'tier': 'open-source'},
}

# Print comparison table
print(f"{'Model':<22} {'Provider':<12} {'Context':>10} {'$/1M in':>9} {'$/1M out':>10} {'Tier'}")
print('-' * 78)
for model, spec in MODEL_SPECS.items():
    ctx = f"{spec['context_k']}k"
    print(f"{model:<22} {spec['provider']:<12} {ctx:>10} ${spec['input_per_1m']:>8.3f} ${spec['output_per_1m']:>9.3f} {spec['tier']}")

## 2. API Connectivity Tests — All Three Providers

In [ ]:
TEST_PROMPT = 'Reply with exactly: "OK"'

# OpenAI
try:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': TEST_PROMPT}],
        max_tokens=5
    )
    print(f"✅ OpenAI (gpt-4o-mini): {r.choices[0].message.content!r}")
except Exception as e:
    print(f"❌ OpenAI: {e}")

# Anthropic
try:
    import anthropic
    ac = anthropic.Anthropic()
    r = ac.messages.create(
        model='claude-3-haiku-20240307',
        max_tokens=5,
        messages=[{'role': 'user', 'content': TEST_PROMPT}]
    )
    print(f"✅ Anthropic (claude-3-haiku): {r.content[0].text!r}")
except ImportError:
    print("📌 Anthropic: pip install anthropic")
except Exception as e:
    print(f"❌ Anthropic: {type(e).__name__}")

# Google Gemini
try:
    import google.generativeai as genai
    genai.configure(api_key=os.getenv('GOOGLE_API_KEY', ''))
    gm = genai.GenerativeModel('gemini-1.5-flash')
    r = gm.generate_content(TEST_PROMPT)
    print(f"✅ Google Gemini (flash): {r.text!r}")
except ImportError:
    print("📌 Google: pip install google-generativeai")
except Exception as e:
    print(f"❌ Google Gemini: {type(e).__name__}")

## 3. Cost Calculator — How Much Does Each Model Cost for My Task?

In [ ]:
def calc_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    spec = MODEL_SPECS.get(model)
    if not spec: return -1
    return (input_tokens * spec['input_per_1m'] / 1e6) + \
           (output_tokens * spec['output_per_1m'] / 1e6)

# Scenario: 10k agent runs, avg 2000 input + 500 output tokens per call
INPUT_TOKENS  = 2000
OUTPUT_TOKENS = 500
N_RUNS        = 10_000

print(f"Cost comparison: {N_RUNS:,} runs × ({INPUT_TOKENS} input + {OUTPUT_TOKENS} output tokens)")
print(f"{'Model':<22} {'Per call':>10} {'10k runs':>12} {'1M runs':>12}")
print('-' * 60)
for model, spec in MODEL_SPECS.items():
    per_call      = calc_cost(model, INPUT_TOKENS, OUTPUT_TOKENS)
    total_10k     = per_call * N_RUNS
    total_1m      = per_call * 1_000_000
    print(f"{model:<22} ${per_call:>9.4f} ${total_10k:>11.2f} ${total_1m:>11.0f}")

## 4. Same Prompt, Different Models — via LiteLLM

In [ ]:
# LiteLLM normalizes APIs — one function for all providers
try:
    import litellm
    litellm.set_verbose = False
    
    PROMPT = 'In one sentence: what is the main advantage of transformer models over RNNs?'
    
    models_to_test = [
        'gpt-4o-mini',
        # 'claude-3-haiku-20240307',    # Uncomment with ANTHROPIC_API_KEY
        # 'gemini/gemini-1.5-flash',    # Uncomment with GOOGLE_API_KEY
    ]
    
    for model_id in models_to_test:
        try:
            t = time.perf_counter()
            r = litellm.completion(
                model=model_id,
                messages=[{'role': 'user', 'content': PROMPT}],
                max_tokens=60
            )
            latency = (time.perf_counter() - t) * 1000
            answer = r.choices[0].message.content
            print(f"\n[{model_id}] ({latency:.0f}ms)")
            print(f"  {answer}")
        except Exception as e:
            print(f"[{model_id}] Error: {type(e).__name__}")
except ImportError:
    print("📌 pip install litellm   then: litellm.completion(model='gpt-4o-mini', messages=[...])")

## 5. Context Window Visualizer

In [ ]:
# Visualize what fits in each model's context
COMPARISONS = [
    ('GPT-4o/mini',      128_000),
    ('Claude 3.5',       200_000),
    ('Gemini 1.5 Flash', 1_000_000),
    ('Gemini 1.5 Pro',   2_000_000),
]

# Approximate sizes of common content types
CONTENT_SIZES = {
    '1 page text':      500,
    '1 book (~300p)':   150_000,
    '1 codebase (med)': 500_000,
    'Linux kernel':     17_000_000,
}

print(f"{'Model':<20} {'Context (k)':>12} {'Pages':>8} {'Books':>8} {'Med Codebase'}")
print('-' * 65)
for name, ctx in COMPARISONS:
    pages = ctx // 500
    books = ctx // 150_000
    fits_codebase = '✅' if ctx >= 500_000 else '❌'
    print(f"{name:<20} {ctx//1000:>10}k {pages:>8,} {books:>8} {fits_codebase:>12}")

print('\nContent size reference:')
for item, tokens in CONTENT_SIZES.items():
    print(f"  {item:<25} ≈ {tokens:>10,} tokens")
print()
print('→ Only Gemini 1.5 Pro (2M) can hold the Linux kernel in one call')

## 📌 Summary

- **MODEL_SPECS dict**: centralized spec reference — context, price, tier
- **Use LiteLLM** for provider-agnostic model comparison with one API
- **Cost scales linearly** — mini/flash models are 10-50× cheaper than flagships
- **Context window matters only when you need it** — most tasks use < 5k tokens
- **Benchmark rankings shift** — always test on your specific task, not just leaderboards